In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

api_key = os.environ["OPENAI_API_KEY"]
print("API key loaded successfully")

Note: The commented codes in this notebook are the granular codes/agents that are used inside the other agents.

## A. Define DOI and STAC links

In [ ]:
stac_url: str = "https://earth.gov/ghgcenter/api/stac"
stac_collection_id: str = "vulcan-ffco2-yeargrid-v4"
publication_doi: str = "https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2020JD032974"

### Story Teller Agent

In [ ]:
from story_teller_agent import StoryTellerAgent, StoryTellerAgentConfig, StoryTellerAgentInputSchema, StoryTellerAgentOutputSchema

story_teller_config = StoryTellerAgentConfig(api_key=api_key)
story_teller_input: StoryTellerAgentInputSchema = StoryTellerAgentInputSchema(
  urls=[publication_doi],
  collection_ids=[stac_collection_id]
)
story_teller_agent: StoryTellerAgent = StoryTellerAgent(config=story_teller_config)
story_teller_output: StoryTellerAgentOutputSchema = await story_teller_agent.arun(story_teller_input)


In [ ]:
story_teller_output.story_mdx

## Detail Breakdown

## 1 + 2. download the stac collection and items

## Multiple Collection Items and Multiple URLs scraping

In [ ]:
from helper import scrape_text_from_urls, get_collection_items
from typing import List
from data_types import CollectionItem

urls: List[str] = [publication_doi]
stac_collection_ids: List[str] = [stac_collection_id, stac_collection_id]


In [ ]:
collection_items: List[CollectionItem] = get_collection_items(stac_collection_ids)

In [ ]:
scraped_text: str = await scrape_text_from_urls(urls)


## 3. Data Scount Agent 

### a. Subagent: Relevant data filter component

In [ ]:
# from relevant_data_filter_agent import get_relevant_data

# relevant_data: List[CollectionItem] = await get_relevant_data(api_key=api_key, literature_context=scraped_text, stac_data=collection_items)

In [ ]:
# print("number of collection_items: ", len(collection_items), "\nnumber of relevant_data: ", len(relevant_data))

In [ ]:
# json_relevant_data = [rd.model_dump_json() for rd in relevant_data]
# print(json_relevant_data)

The above are commented as these subagents are part of the Data Scout Agent already.

### b. Subagent: Data relationship builder component

In [ ]:
#NA

### c. Main agent: DATA SCOUT AGENT

In [ ]:
from data_scout_agent import DataScoutAgent, DataScoutAgentInputSchema, DataScoutAgentOutputSchema, DataScoutAgentConfig

data_scout_input: DataScoutAgentInputSchema = DataScoutAgentInputSchema(
  literature=scraped_text,
  collection_items=collection_items
)
data_scout_agent_config: DataScoutAgentConfig = DataScoutAgentConfig(api_key=api_key)
data_scout_agent: DataScoutAgent = DataScoutAgent(config=data_scout_agent_config)
data_scout_agent_output: DataScoutAgentOutputSchema = await data_scout_agent.arun(data_scout_input)

In [ ]:
# mock_relevant_collection_items = [CollectionItem(collection_id='vulcan-ffco2-yeargrid-v4', collection_description='Annual (2010 - 2021), 1 km resolution estimates of carbon dioxide emissions from fossil fuels and cement production over the contiguous United States, version 4.0', collection_title='Vulcan Fossil Fuel CO₂ Emissions v4.0', item_id='vulcan-ffco2-yeargrid-v4-2011', location=[-96, 37], date='2011-01-01', location_name='Chautauqua County, Kansas, United States', item_description='Estimated total annual ffCO₂ emissions coming from railroads., Estimated total annual CO₂ emissions from fossil fuel combustion (ffCO₂) across all sectors., Cloud optimized default layer to display on map', item_title='Total Airport CO₂ Emissions, Total Cement CO₂ Emissions, Total Commercial Marine Vessels CO₂ Emissions, Total Commercial CO₂ Emissions, Total Powerplants CO₂ Emissions, Total Industrial CO₂ Emissions, Total Nonroad CO₂ Emissions, Total Onroad CO₂ Emissions, Total Residential CO₂ Emissions, Total Railroad CO₂ Emissions, Total of all sectors CO₂ Emissions, Default COG Layer, Rendered preview')]

In [ ]:
print("Number of input collections items: ", len(data_scout_input.collection_items))
print("Number of relevant collections items: ", len(data_scout_agent_output.relevant_collection))

## 4. Script writer agent

### a. Subagent: Script blueprint builder agent

In [ ]:
# from script_blueprint_builder_agent import ScriptBlueprintBuilderAgent, ScriptBlueprintBuilderAgentConfig, ScriptBlueprintBuilderAgentInputSchema, ScriptBlueprintBuilderAgentOutputSchema

# config = ScriptBlueprintBuilderAgentConfig(api_key=api_key)

# script_blueprint_input: ScriptBlueprintBuilderAgentInputSchema = ScriptBlueprintBuilderAgentInputSchema(
#   literature_text=scraped_text,
#   collection_items=data_scout_agent_output.relevant_collection
# )

# script_blueprint_agent: ScriptBlueprintBuilderAgent = ScriptBlueprintBuilderAgent(config=config)

# script_blueprint_output: ScriptBlueprintBuilderAgentOutputSchema = await script_blueprint_agent.arun(script_blueprint_input)

In [ ]:
# mock_story_blueprint: str = """
#   **Title:** Mapping the Invisible Threat: CO₂ Emissions Across America's Heartland  \n**Target Audience:** General Public  \n**Logline:** Exploring the distribution and impact of carbon dioxide emissions across various sectors in the United States in 2011, centered in Chautauqua County, Kansas.\n\n**Script Body:**  \n\n- **Section Header:** Introduction  \n  - **Context/Scene Setting:** We're standing in the vast plains of Chautauqua County, Kansas, a central node in America's geographical map and now the focal point of a crucial scientific narrative—carbon dioxide emissions. The date is 2011, and we're about to uncover the unseen clouds hovering above and beyond, holding the key to our environmental future.  \n  - **Visual Description:** The camera pans over expansive landscapes, segues into dynamic graphics showing CO₂ emissions rising from different sectors like power plants, railroads, and residential areas.  \n  - **Narrative Text:** In 2011, the release of carbon dioxide from fossil fuel combustion and cement production was meticulously assessed through the Vulcan Project, bringing a granular view of emissions across the U.S. landscape. Here in the heart of the country, emissions from various sectors weave an invisible tapestry affecting climate and policy discussions worldwide.\n\n- **Section Header:** The Data Event  \n  - **Context/Scene Setting:** We now dive into a composite image of emission sources: from the steam billowing out of a power plant stack to the smoke trailing a passing freight train in Chautauqua County.  \n  - **Visual Description:** Time-lapse sequences of emissions from different sources such as an airport, cement factory, and commercial and industrial sectors are showcased alongside data tables and graphics.  \n  - **Narrative Text:** According to the Vulcan v4.0 data, the total carbon emissions recorded in 2011 reveal multiple avenues of fossil fuel CO₂ deposit: from power plants producing energy to keep our homes lit, to airplanes destined for the skies, each contributing to a growing global challenge. The center of U.S. FFCO₂ emissions sat over Missouri, but the data from Kansas serves as a microcosm of the whole.\n\n- **Section Header:** The Broader Impact  \n  - **Context/Scene Setting:** Shift focus to global maps displaying CO₂ trajectories and aggregate emissions, showing Kansas as a part of a larger, interconnected system.  \n  - **Visual Description:** Use of visual aids such as 3D graphs and world maps to illustrate the flow and concentration of emissions impacting global climate patterns.  \n  - **Narrative Text:** What happens here in Kansas is not isolated. Our area's emissions trend north in colder months and descend south in warmer, intensifying the ever-complex climate phenomenon. The data—from industrial giants to household culprits—illustrates a broader implication, pushing scientific communities to pioneer solutions and encourage sustainable practices.\n\n- **Section Header:** Conclusion  \n  - **Context/Scene Setting:** A return to the Kansas plains, silhouetted against a setting sun. There is hope on the horizon carried on winds of change.  \n  - **Visual Description:** End with a montage of renewable energy installations juxtaposed with the traditional power stations, subtly suggesting a transition underway.  \n  - **Narrative Text:** As we look to the future, understanding these emission patterns equips us with insight, allowing for informed decision-making and action. Solutions sprouting across sectors nurture optimism, aligning economic goals with ecological necessities. The journey starts here, in Chautauqua County, as America maps its pathway towards a cleaner tomorrow.
# """

### b. Subagent: Script builder agent

In [ ]:
# from script_builder_agent import ScriptBuilderAgent, ScriptBuilderAgentConfig, ScriptBuilderAgentInputSchema, ScriptBuilderAgentOutputSchema

# script_builder_config = ScriptBuilderAgentConfig(api_key=api_key)
# script_builder_input = ScriptBuilderAgentInputSchema(
#   narrative_blueprint=script_blueprint_output.script_blueprint
# )
# script_builder_agent: ScriptBuilderAgent = ScriptBuilderAgent(config=script_builder_config)
# script_builder_output: ScriptBuilderAgentOutputSchema = await script_builder_agent.arun(script_builder_input)

In [ ]:
# mock_story_draft: str = """
#   <article>\n<h1>Mapping the Invisible Threat: CO₂ Emissions Across America\'s Heartland</h1>\n<section class="chapter">\n<h3>Introduction</h3>\n<p>We’re standing in the vast plains of <strong>Chautauqua County, Kansas</strong>, a central node in America’s geographical map and now the focal point of a crucial scientific narrative—<strong>carbon dioxide emissions</strong>. The date is <strong>2011</strong>, and we’re about to uncover the unseen clouds hovering above and beyond, holding the key to our environmental future.</p>\n<p>The camera pans over expansive landscapes, segues into dynamic graphics showing <strong>CO₂ emissions</strong> rising from different sectors like power plants, railroads, and residential areas.</p>\n<p>In 2011, the release of carbon dioxide from fossil fuel combustion and cement production was meticulously assessed through the <strong>Vulcan Project</strong>, bringing a granular view of emissions across the U.S. landscape. Here in the heart of the country, emissions from various sectors weave an invisible tapestry affecting climate and policy discussions worldwide.</p>\n</section>\n<section class="chapter">\n<h3>The Data Event</h3>\n<p>We now dive into a composite image of emission sources: from the steam billowing out of a power plant stack to the smoke trailing a passing freight train in Chautauqua County.</p>\n<p>Time-lapse sequences of emissions from different sources such as an airport, cement factory, and commercial and industrial sectors are showcased alongside data tables and graphics.</p>\n<p>According to the <strong>Vulcan v4.0 data</strong>, the total carbon emissions recorded in 2011 reveal multiple avenues of fossil fuel CO₂ deposit: from power plants producing energy to keep our homes lit, to airplanes destined for the skies, each contributing to a growing global challenge. The center of U.S. FFCO₂ emissions sat over Missouri, but the data from Kansas serves as a microcosm of the whole.</p>\n</section>\n<section class="chapter">\n<h3>The Broader Impact</h3>\n<p>We shift focus to global maps displaying CO₂ trajectories and aggregate emissions, showcasing Kansas as a part of a larger, interconnected system.</p>\n<p>Visual aids such as 3D graphs and world maps illustrate the flow and concentration of emissions impacting global climate patterns.</p>\n<p>What happens here in Kansas is not isolated. Our area\'s emissions trend north in colder months and descend south in warmer, intensifying the ever-complex climate phenomenon. The data—from industrial giants to household culprits—illustrates a broader implication, pushing scientific communities to pioneer solutions and encourage sustainable practices.</p>\n</section>\n<section class="chapter">\n<h3>Conclusion</h3>\n<p>A return to the Kansas plains, silhouetted against a setting sun. There is hope on the horizon carried on winds of change.</p>\n<p>Ending with a montage of renewable energy installations juxtaposed with the traditional power stations subtly suggests a transition underway.</p>\n<p>As we look to the future, understanding these emission patterns equips us with insight, allowing for informed decision-making and action. Solutions sprouting across sectors nurture optimism, aligning economic goals with ecological necessities. The journey starts here, in Chautauqua County, as America maps its pathway towards a cleaner tomorrow.</p>\n</section>\n</article>
# """

### c. Subagent: Script Critique agent

In [ ]:
## For this initial implementation, lets assume that we do not need critique.
## TODO: Need to implement this using deep eval

#### d. Main Agent: SCRIPT WRITER Agent

In [ ]:
from script_writer_agent import ScriptWriterAgent, ScriptWriterAgentConfig, ScriptWriterAgentInputSchema, ScriptWriterAgentOutputSchema

script_writer_config = ScriptWriterAgentConfig(api_key=api_key)

script_writer_input: ScriptWriterAgentInputSchema = ScriptWriterAgentInputSchema(
  literature_text=scraped_text,
  collection_items=data_scout_agent_output.relevant_collection
)
script_writer_agent: ScriptWriterAgent = ScriptWriterAgent(config=script_writer_config)
script_writer_output: ScriptWriterAgentOutputSchema = await script_writer_agent.arun(script_writer_input)

In [ ]:
# script_writer_output.script

## 5. Data Injection Component

In [ ]:
from data_injection_agent import DataInjectionAgent, DataInjectionAgentConfig, DataInjectionAgentInputSchema, DataInjectionAgentOutputSchema

data_injection_agent_config: DataInjectionAgentConfig = DataInjectionAgentConfig(api_key=api_key)

data_injection_agent_input: DataInjectionAgentInputSchema = DataInjectionAgentInputSchema(
  script=script_writer_output.script,
  collection_items=data_scout_agent_output.relevant_collection
)

data_injection_agent: DataInjectionAgent = DataInjectionAgent(config=data_injection_agent_config)
data_injected_script: DataInjectionAgentOutputSchema = await data_injection_agent.arun(data_injection_agent_input)


In [ ]:
mock_story_with_data: str = """
  <article>\n<h1>Mapping the Invisible Threat: CO₂ Emissions Across America\'s Heartland</h1>\n<section class="chapter">\n<h3>Introduction</h3>\n<p>We’re standing in the vast plains of <strong>Chautauqua County, Kansas</strong>, a central node in America’s geographical map and now the focal point of a crucial scientific narrative—<strong>carbon dioxide emissions</strong>. The date is <strong>2011</strong>, and we’re about to uncover the unseen clouds hovering above and beyond, holding the key to our environmental future.</p>\n<p>The camera pans over expansive landscapes, segues into dynamic graphics showing <strong>CO₂ emissions</strong> rising from different sectors like power plants, railroads, and residential areas.</p>\n<p>In 2011, the release of carbon dioxide from fossil fuel combustion and cement production was meticulously assessed through the <strong>Vulcan Project</strong>, bringing a granular view of emissions across the U.S. landscape. Here in the heart of the country, emissions from various sectors weave an invisible tapestry affecting climate and policy discussions worldwide.</p>\n</section>\n<section class="chapter">\n<h3>The Data Event</h3>\n<p>We now dive into a composite image of emission sources: from the steam billowing out of a power plant stack to the smoke trailing a passing freight train in Chautauqua County.</p>\n<p>Time-lapse sequences of emissions from different sources such as an airport, cement factory, and commercial and industrial sectors are showcased alongside data tables and graphics.</p>\n<p>According to the <strong>Vulcan v4.0 data</strong>, the total carbon emissions recorded in 2011 reveal multiple avenues of fossil fuel CO₂ deposit: from power plants producing energy to keep our homes lit, to airplanes destined for the skies, each contributing to a growing global challenge. The center of U.S. FFCO₂ emissions sat over Missouri, but the data from Kansas serves as a microcosm of the whole.</p>\n<xml>\n<GroupedData>\n  <DataBlock>\n    <CollectionId>vulcan-ffco2-yeargrid-v4</CollectionId>\n    <ItemId>vulcan-ffco2-yeargrid-v4-2011</ItemId>\n    <Datetime>2011-01-01</Datetime>\n    <center>\n      <lon>-96</lon>\n      <lat>37</lat>\n    </center>\n    <zoom>5</zoom>\n    <content>Total Carbon Dioxide emissions for various sectors recorded in 2011. This includes emissions from power plants, railroads, and other sectors contributing to the broader emission narrative.</content>\n  </DataBlock>\n</GroupedData>\n</xml>\n</section>\n<section class="chapter">\n<h3>The Broader Impact</h3>\n<p>We shift focus to global maps displaying CO₂ trajectories and aggregate emissions, showcasing Kansas as a part of a larger, interconnected system.</p>\n<p>Visual aids such as 3D graphs and world maps illustrate the flow and concentration of emissions impacting global climate patterns.</p>\n<p>What happens here in Kansas is not isolated. Our area\'s emissions trend north in colder months and descend south in warmer, intensifying the ever-complex climate phenomenon. The data—from industrial giants to household culprits—illustrates a broader implication, pushing scientific communities to pioneer solutions and encourage sustainable practices.</p>\n</section>\n<section class="chapter">\n<h3>Conclusion</h3>\n<p>A return to the Kansas plains, silhouetted against a setting sun. There is hope on the horizon carried on winds of change.</p>\n<p>Ending with a montage of renewable energy installations juxtaposed with the traditional power stations subtly suggests a transition underway.</p>\n<p>As we look to the future, understanding these emission patterns equips us with insight, allowing for informed decision-making and action. Solutions sprouting across sectors nurture optimism, aligning economic goals with ecological necessities. The journey starts here, in Chautauqua County, as America maps its pathway towards a cleaner tomorrow.</p>\n</section>\n</article>
"""

## 6. MDX Builder Component

In [ ]:
from mdx_builder_agent import MDXBuilderAgent, MDXBuilderAgentConfig, MDXBuilderAgentInputSchema, MDXBuilderAgentOutputSchema

mdx_builder_config: MDXBuilderAgentConfig = MDXBuilderAgentConfig(api_key=api_key)

mdx_builder_input_schema: MDXBuilderAgentInputSchema = MDXBuilderAgentInputSchema(
    story_script=data_injected_script.script_with_data
)

mdx_builder_agent: MDXBuilderAgent = MDXBuilderAgent(mdx_builder_config)
mdx_builder_output: MDXBuilderAgentOutputSchema = await mdx_builder_agent.arun(mdx_builder_input_schema)

mdx_story: str = mdx_builder_output.story_mdx

In [ ]:
mdx_story

Tests with MDX validator:

In [ ]:
mock_mdx_story = """
  <Block>\n  <Prose>\n    # Mapping the Invisible Threat: CO₂ Emissions Across America's Heartland\n    \n    ## Introduction\n    \n    We’re standing in the vast plains of **Chautauqua County, Kansas**, a central node in America’s geographical map and now the focal point of a crucial scientific narrative—**carbon dioxide emissions**. The date is **2011**, and we’re about to uncover the unseen clouds hovering above and beyond, holding the key to our environmental future.\n    \n    The camera pans over expansive landscapes, segues into dynamic graphics showing **CO₂ emissions** rising from different sectors like power plants, railroads, and residential areas.\n\n    In 2011, the release of carbon dioxide from fossil fuel combustion and cement production was meticulously assessed through the **Vulcan Project**, bringing a granular view of emissions across the U.S. landscape. Here in the heart of the country, emissions from various sectors weave an invisible tapestry affecting climate and policy discussions worldwide.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## The Data Event\n\n    We now dive into a composite image of emission sources: from the steam billowing out of a power plant stack to the smoke trailing a passing freight train in Chautauqua County.\n    \n    Time-lapse sequences of emissions from different sources such as an airport, cement factory, and commercial and industrial sectors are showcased alongside data tables and graphics.\n\n    According to the **Vulcan v4.0 data**, the total carbon emissions recorded in 2011 reveal multiple avenues of fossil fuel CO₂ deposit: from power plants producing energy to keep our homes lit, to airplanes destined for the skies, each contributing to a growing global challenge. The center of U.S. FFCO₂ emissions sat over Missouri, but the data from Kansas serves as a microcosm of the whole.\n  </Prose>\n</Block>\n\n<ScrollytellingBlock>\n  <Chapter\n    center={[-96, 37]}\n    zoom={5}\n    datasetId='vulcan-ffco2-yeargrid-v4'\n    layerId='vulcan-ffco2-yeargrid-v4-2011'\n    datetime='2011-01-01'\n  >\n    <Prose>\n      Total Carbon Dioxide emissions for various sectors recorded in 2011. This includes emissions from power plants, railroads, and other sectors contributing to the broader emission narrative.\n    </Prose>\n  </Chapter>\n</ScrollytellingBlock>\n\n<Block>\n  <Prose>\n    ## The Broader Impact\n\n    We shift focus to global maps displaying CO₂ trajectories and aggregate emissions, showcasing Kansas as a part of a larger, interconnected system.\n    \n    Visual aids such as 3D graphs and world maps illustrate the flow and concentration of emissions impacting global climate patterns.\n\n    What happens here in Kansas is not isolated. Our area's emissions trend north in colder months and descend south in warmer, intensifying the ever-complex climate phenomenon. The data—from industrial giants to household culprits—illustrates a broader implication, pushing scientific communities to pioneer solutions and encourage sustainable practices.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## Conclusion\n\n    A return to the Kansas plains, silhouetted against a setting sun. There is hope on the horizon carried on winds of change.\n    \n    Ending with a montage of renewable energy installations juxtaposed with the traditional power stations subtly suggests a transition underway.\n\n    As we look to the future, understanding these emission patterns equips us with insight, allowing for informed decision-making and action. Solutions sprouting across sectors nurture optimism, aligning economic goals with ecological necessities. The journey starts here, in Chautauqua County, as America maps its pathway towards a cleaner tomorrow.\n  </Prose>\n</Block>
"""

In [ ]:
mock_mdx_story_broken = """
  <Block>\n    # Mapping the Invisible Threat: CO₂ Emissions Across America's Heartland\n    \n    ## Introduction\n    \n    We’re standing in the vast plains of **Chautauqua County, Kansas**, a central node in America’s geographical map and now the focal point of a crucial scientific narrative—**carbon dioxide emissions**. The date is **2011**, and we’re about to uncover the unseen clouds hovering above and beyond, holding the key to our environmental future.\n    \n    The camera pans over expansive landscapes, segues into dynamic graphics showing **CO₂ emissions** rising from different sectors like power plants, railroads, and residential areas.\n\n    In 2011, the release of carbon dioxide from fossil fuel combustion and cement production was meticulously assessed through the **Vulcan Project**, bringing a granular view of emissions across the U.S. landscape. Here in the heart of the country, emissions from various sectors weave an invisible tapestry affecting climate and policy discussions worldwide.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## The Data Event\n\n    We now dive into a composite image of emission sources: from the steam billowing out of a power plant stack to the smoke trailing a passing freight train in Chautauqua County.\n    \n    Time-lapse sequences of emissions from different sources such as an airport, cement factory, and commercial and industrial sectors are showcased alongside data tables and graphics.\n\n    According to the **Vulcan v4.0 data**, the total carbon emissions recorded in 2011 reveal multiple avenues of fossil fuel CO₂ deposit: from power plants producing energy to keep our homes lit, to airplanes destined for the skies, each contributing to a growing global challenge. The center of U.S. FFCO₂ emissions sat over Missouri, but the data from Kansas serves as a microcosm of the whole.\n  </Prose>\n</Block>\n\n<ScrollytellingBlock>\n  <Chapter\n    center={[-96, 37]}\n    zoom={5}\n    datasetId='vulcan-ffco2-yeargrid-v4'\n    layerId='vulcan-ffco2-yeargrid-v4-2011'\n    datetime='2011-01-01'\n  >\n    <Prose>\n      Total Carbon Dioxide emissions for various sectors recorded in 2011. This includes emissions from power plants, railroads, and other sectors contributing to the broader emission narrative.\n    </Prose>\n  </Chapter>\n</ScrollytellingBlock>\n\n<Block>\n  <Prose>\n    ## The Broader Impact\n\n    We shift focus to global maps displaying CO₂ trajectories and aggregate emissions, showcasing Kansas as a part of a larger, interconnected system.\n    \n    Visual aids such as 3D graphs and world maps illustrate the flow and concentration of emissions impacting global climate patterns.\n\n    What happens here in Kansas is not isolated. Our area's emissions trend north in colder months and descend south in warmer, intensifying the ever-complex climate phenomenon. The data—from industrial giants to household culprits—illustrates a broader implication, pushing scientific communities to pioneer solutions and encourage sustainable practices.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ## Conclusion\n\n    A return to the Kansas plains, silhouetted against a setting sun. There is hope on the horizon carried on winds of change.\n    \n    Ending with a montage of renewable energy installations juxtaposed with the traditional power stations subtly suggests a transition underway.\n\n    As we look to the future, understanding these emission patterns equips us with insight, allowing for informed decision-making and action. Solutions sprouting across sectors nurture optimism, aligning economic goals with ecological necessities. The journey starts here, in Chautauqua County, as America maps its pathway towards a cleaner tomorrow.\n  </Prose>\n</Block>
"""

In [ ]:
from helper import MDXValidator

validator = MDXValidator()
valid, feedback = validator.validate_mdx(mock_mdx_story_broken)
print(valid)


In [ ]:
from mdx_builder_agent import MDXBuilderAgent, MDXBuilderAgentConfig, MDXBuilderAgentInputSchema, MDXBuilderAgentOutputSchema

mdx_builder_config: MDXBuilderAgentConfig = MDXBuilderAgentConfig(api_key=api_key)

mdx_builder_input_schema: MDXBuilderAgentInputSchema = MDXBuilderAgentInputSchema(
    # story_script=data_injected_script.script_with_data
    story_script=mock_story_with_data,
    feedback=feedback,
    previous_generated_mdx=mock_mdx_story_broken
)

mdx_builder_agent: MDXBuilderAgent = MDXBuilderAgent(mdx_builder_config)
mdx_builder_output: MDXBuilderAgentOutputSchema = await mdx_builder_agent.arun(mdx_builder_input_schema)

corrected_mdx_story: str = mdx_builder_output.story_mdx


In [ ]:
from helper import MDXValidator

validator = MDXValidator()
valid, feedback = validator.validate_mdx(corrected_mdx_story)
print(valid)

In [ ]:
print(mock_mdx_story_broken)
print(corrected_mdx_story)